In [116]:
from pathlib import Path
import numpy as np
import pandas as pd

In [117]:
PROJECT_DIR = Path.cwd().resolve().parent
DATA_DIR    = PROJECT_DIR / 'data'
SRC_DIR     = PROJECT_DIR / 'src'
EXTRACT_DIR = PROJECT_DIR / 'processed' / 'extraction'
REPORT_DIR  = PROJECT_DIR / 'reports'

print('PROJECT_DIR:', PROJECT_DIR)
print('DATA_DIR:', DATA_DIR)
print('SRC_DIR:', SRC_DIR)

PROJECT_DIR: /Users/oneash/ONEASH_local/Coding/Delirium-Prediction/Parkinson
DATA_DIR: /Users/oneash/ONEASH_local/Coding/Delirium-Prediction/Parkinson/data
SRC_DIR: /Users/oneash/ONEASH_local/Coding/Delirium-Prediction/Parkinson/src


In [118]:
d_items = pd.read_csv(DATA_DIR / 'd_items.csv') # itemid-label 정보
d_labitems = pd.read_csv( DATA_DIR / 'd_labitems.csv') # lab itemid-label 정보

VARIABLE_CATALOG_PATH = SRC_DIR / 'extraction_variable_catalog.csv' # 사용할 변수 카테고리
variable_catalog = pd.read_csv(VARIABLE_CATALOG_PATH)

display(d_items.head())
display(d_labitems.head())
display(variable_catalog.head())

,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
0,220001,Problem List,Problem List,chartevents,General,NaN,Text,NaN,NaN
1,220003,ICU Admission date,ICU Admission date,datetimeevents,ADT,NaN,Date and time,NaN,NaN
2,220045,Heart Rate,HR,chartevents,Routine Vital Signs,bpm,Numeric,NaN,NaN
3,220046,Heart rate Alarm - High,HR Alarm - High,chartevents,Alarms,bpm,Numeric,NaN,NaN
4,220047,Heart Rate Alarm - Low,HR Alarm - Low,chartevents,Alarms,bpm,Numeric,NaN,NaN


,itemid,label,fluid,category
0,50801,Alveolar-arterial Gradient,Blood,Blood Gas
1,50802,Base Excess,Blood,Blood Gas
2,50803,"Calculated Bicarbonate, Whole Blood",Blood,Blood Gas
3,50804,Calculated Total CO2,Blood,Blood Gas
4,50805,Carboxyhemoglobin,Blood,Blood Gas


,source_table,itemid,category,label,feature_name,unit_or_window,type,transform_note,binning
0,patients,NaN,NaN,anchor_age,age,NaN,numeric,NaN,static
1,patients,NaN,NaN,gender,gender,NaN,binary,NaN,static
2,chartevents,228332.0,NaN,Delirium assessment,delirium,Positive/Negative/UTA,binary,"Positive=1, Negative=0, UTA/other=NaN; primary...",at least once
3,chartevents,228096.0,NaN,Richmond-RAS Scale,rass,-5 to +4,ordinal,value (text) -> integer,aggregation
4,chartevents,220739.0,NaN,GCS - Eye Opening,gcs_eye,1 to 4,ordinal,use valuenum,aggregation


## 1. Cohort

모든 이벤트를 ICU stay 시간 안으로 제한


In [119]:
patients   = pd.read_csv(DATA_DIR / 'patients.csv')
admissions = pd.read_csv(DATA_DIR / 'admissions.csv')
icustays   = pd.read_csv(DATA_DIR / 'icustays.csv')
services   = pd.read_csv(DATA_DIR / 'services.csv')


In [120]:
# 입원 및 ICU 입실/퇴실 시간 -> datetime 형식
for col in ['intime', 'outtime']:
    icustays[col] = pd.to_datetime(icustays[col], errors='coerce') # 변환 실패 -> NaT
for col in ['admittime', 'dischtime', 'deathtime']:
    if col in admissions.columns:
        admissions[col] = pd.to_datetime(admissions[col], errors='coerce')
services['transfertime'] = pd.to_datetime(services['transfertime'], errors='coerce')


In [121]:
# ICU stay 단위로 환자 정보, 입원 정보, ICU 입실 시점 service 정보를 결합
adm_pat_icu = (
    icustays
    .merge(patients[['subject_id', 'gender', 'anchor_age', 'anchor_year', 'anchor_year_group', 'dod']], on='subject_id', how='left')
    .merge(admissions[['subject_id', 'hadm_id', 'admittime', 'dischtime', 'deathtime', 'admission_type', 'race']], on=['subject_id', 'hadm_id'], how='left')
)

specialty_by_stay = (
    services[['subject_id', 'hadm_id', 'transfertime', 'curr_service']]
    .dropna(subset=['curr_service'])
    .merge(
        icustays[['subject_id', 'hadm_id', 'stay_id', 'intime']],
        on=['subject_id', 'hadm_id'],
        how='inner',
    )
    .loc[lambda df: df['transfertime'].le(df['intime'])]
    .sort_values(['stay_id', 'transfertime'])
    .drop_duplicates('stay_id', keep='last')
    [['stay_id', 'curr_service']]
    .rename(columns={'curr_service': 'specialty'})
)
adm_pat_icu = adm_pat_icu.merge(specialty_by_stay, on='stay_id', how='left')

adm_pat_icu['icu_los_hours'] = (adm_pat_icu['outtime'] - adm_pat_icu['intime']).dt.total_seconds() / 3600
display(adm_pat_icu.head())
adm_pat_icu.to_csv(EXTRACT_DIR / 'adm_pat_icu.csv', index=False)


,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,gender,anchor_age,anchor_year,anchor_year_group,dod,admittime,dischtime,deathtime,admission_type,race,specialty,icu_los_hours
0,10018328,23786647,31269608,Neuro Stepdown,Neuro Stepdown,2154-04-24 23:03:44,2154-05-02 15:55:21,7.702512,F,83,2154,2014 - 2016,2158-12-14,2154-04-24 03:15:00,2154-05-03 14:00:00,NaT,SURGICAL SAME DAY ADMISSION,WHITE,NSURG,184.860278
1,10018328,28252562,37006782,Neuro Intermediate,Neuro Intermediate,2158-02-08 22:56:08,2158-02-13 23:58:23,5.043229,F,83,2154,2014 - 2016,2158-12-14,2158-02-08 22:55:00,2158-02-18 17:30:00,NaT,OBSERVATION ADMIT,WHITE,NSURG,121.037500
2,10050755,20050796,37743005,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2134-02-26 01:03:02,2134-02-26 04:27:45,0.142164,M,77,2126,2008 - 2010,2134-02-26,2134-02-25 18:41:00,2134-02-26 01:26:00,2134-02-26 01:26:00,OBSERVATION ADMIT,WHITE - RUSSIAN,MED,3.411944
3,10052769,22087051,30336368,Neuro Surgical Intensive Care Unit (Neuro SICU),Neuro Surgical Intensive Care Unit (Neuro SICU),2124-06-16 20:18:49,2124-07-08 14:31:39,21.758912,M,78,2124,2020 - 2022,2124-07-09,2124-04-25 19:33:00,2124-07-09 09:43:00,2124-07-09 09:43:00,URGENT,UNKNOWN,MED,522.213889
4,10052769,22087051,34171709,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2124-05-25 15:48:47,2124-05-26 02:14:53,0.434792,M,78,2124,2020 - 2022,2124-07-09,2124-04-25 19:33:00,2124-07-09 09:43:00,2124-07-09 09:43:00,URGENT,UNKNOWN,MED,10.435000


## 2. CHARTEVENTS 추출

In [122]:
chart_catalog = variable_catalog[variable_catalog['source_table'].eq('chartevents')].copy()
chart_catalog['itemid'] = pd.to_numeric(chart_catalog['itemid'], errors='coerce').astype('int')
chart_catalog.head()

,source_table,itemid,category,label,feature_name,unit_or_window,type,transform_note,binning
2,chartevents,228332,NaN,Delirium assessment,delirium,Positive/Negative/UTA,binary,"Positive=1, Negative=0, UTA/other=NaN; primary...",at least once
3,chartevents,228096,NaN,Richmond-RAS Scale,rass,-5 to +4,ordinal,value (text) -> integer,aggregation
4,chartevents,220739,NaN,GCS - Eye Opening,gcs_eye,1 to 4,ordinal,use valuenum,aggregation
5,chartevents,223900,NaN,GCS - Verbal Response,gcs_verbal,1 to 5,ordinal,use valuenum,aggregation
6,chartevents,223901,NaN,GCS - Motor Response,gcs_motor,1 to 6,ordinal,use valuenum,aggregation


In [123]:
chart_itemids = chart_catalog['itemid'].tolist()
chart_labels  = chart_catalog['label'].tolist()

# 큰 파일이므로 chunk 단위로 read
chart_chunks = []
chart_cols = ['subject_id', 'hadm_id', 'stay_id', 'charttime', 'itemid', 'value', 'valuenum', 'valueuom']
for chunk in pd.read_csv(DATA_DIR / 'chartevents.csv', usecols=chart_cols, chunksize=1_000_000):
    selected = chunk[chunk['itemid'].isin(chart_itemids)].copy()
    if len(selected):
        selected = selected.merge(d_items[['itemid', 'label', 'category', 'unitname']], on='itemid', how='left')
        selected = selected.merge(chart_catalog[['itemid', 'feature_name', 'type']], on='itemid', how='left')
        chart_chunks.append(selected)

In [124]:
chart_selected = pd.concat(chart_chunks, ignore_index=True)

chart_selected['charttime'] = pd.to_datetime(chart_selected['charttime'], errors='coerce')

static_body_features = ['height', 'weight']
static_body_mask = chart_selected['feature_name'].isin(static_body_features)

# Height/weight are static body measurements. In MIMIC they are often
# charted minutes to hours before ICU intime, so use the first record within each
# hospital admission instead of requiring the event time to fall inside the ICU stay.
# For height only, fall back to the patient's first available height if the current
# hospital admission has no height record.
static_body_first = (
    chart_selected
    .loc[
        static_body_mask
        & chart_selected['hadm_id'].isin(adm_pat_icu['hadm_id'])
        & chart_selected['charttime'].notna()
    ]
    .sort_values(['hadm_id', 'feature_name', 'charttime', 'itemid'])
    .drop_duplicates(['hadm_id', 'feature_name'], keep='first')
    [['hadm_id', 'feature_name', 'charttime', 'itemid', 'value', 'valuenum', 'valueuom', 'label', 'category', 'unitname', 'type']]
)
height_first_by_subject = (
    chart_selected
    .loc[
        chart_selected['feature_name'].eq('height')
        & chart_selected['subject_id'].isin(adm_pat_icu['subject_id'])
        & chart_selected['charttime'].notna()
    ]
    .sort_values(['subject_id', 'charttime', 'itemid'])
    .drop_duplicates(['subject_id'], keep='first')
    [['subject_id', 'feature_name', 'charttime', 'itemid', 'value', 'valuenum', 'valueuom', 'label', 'category', 'unitname', 'type']]
)

chart_selected = chart_selected.merge(adm_pat_icu[['subject_id', 'hadm_id', 'stay_id', 'intime', 'outtime']],
                                      on=['subject_id', 'hadm_id', 'stay_id'], how='inner')
in_icu_window = ((chart_selected['charttime'] >= chart_selected['intime']) &
                 (chart_selected['charttime'] <= chart_selected['outtime']))
static_body_mask = chart_selected['feature_name'].isin(static_body_features)

static_body_for_stays = adm_pat_icu[['subject_id', 'hadm_id', 'stay_id', 'intime', 'outtime']].merge(
    static_body_first,
    on='hadm_id',
    how='inner'
)
height_hadm_ids = set(static_body_first.loc[static_body_first['feature_name'].eq('height'), 'hadm_id'])
height_subject_fallback = adm_pat_icu.loc[
    ~adm_pat_icu['hadm_id'].isin(height_hadm_ids),
    ['subject_id', 'hadm_id', 'stay_id', 'intime', 'outtime']
].merge(
    height_first_by_subject,
    on='subject_id',
    how='inner'
)
static_body_for_stays = pd.concat([static_body_for_stays, height_subject_fallback], ignore_index=True)
# Downstream 12h/bin logic expects an ICU-time row; the value itself still comes from
# the first hospital-admission chart record selected above.
static_body_for_stays['charttime'] = pd.to_datetime(static_body_for_stays['intime'], errors='coerce')

chart_selected = pd.concat(
    [chart_selected.loc[~static_body_mask & in_icu_window].copy(), static_body_for_stays],
    ignore_index=True
)

chart_selected.to_csv(EXTRACT_DIR / 'chart_selected.csv', index=False)

print('chart_selected rows:', len(chart_selected))
print('chart_selected stays:', chart_selected['stay_id'].nunique())
print('chart_itemid counts to be extracted:', len(chart_catalog))
print('chart_itemid counts extracted:', chart_selected.itemid.nunique())
print('chart_feature counts:', chart_selected.feature_name.nunique())
print('static body rows by feature:')
print(chart_selected.loc[chart_selected['feature_name'].isin(static_body_features), 'feature_name'].value_counts())

chart_selected rows: 679679
chart_selected stays: 974
chart_itemid counts to be extracted: 26
chart_itemid counts extracted: 25
chart_feature counts: 19
static body rows by feature:
feature_name
weight    973
height    638
Name: count, dtype: int64


## 3. LABEVENTS 추출

In [125]:
lab_catalog = variable_catalog[variable_catalog['source_table'].eq('labevents')].copy()
lab_catalog['itemid'] = pd.to_numeric(lab_catalog['itemid'], errors='coerce').astype('int')
lab_catalog.head()

,source_table,itemid,category,label,feature_name,unit_or_window,type,transform_note,binning
28,labevents,50931,blood chemistry,Glucose,glucose,mg/dL,numeric,NaN,aggregation
29,labevents,52569,blood chemistry,Glucose,glucose,mg/dL,numeric,NaN,aggregation
30,labevents,50803,blood gas,"Calculated Bicarbonate, Whole Blood",bicarbonate,mEq/L,numeric,NaN,most recent
31,labevents,50893,blood chemistry,"Calcium, Total",calcium,mg/dL,numeric,NaN,most recent
32,labevents,50902,blood chemistry,Chloride,chloride,mEq/L,numeric,NaN,most recent


In [126]:
lab_itemids = lab_catalog['itemid'].tolist()
lab_labels  = lab_catalog['label'].tolist()

# 큰 파일이므로 chunk 단위로 read
lab_chunks = []
lab_cols = ['labevent_id', 'subject_id', 'hadm_id', 'itemid', 'charttime', 'value', 'valuenum', 'valueuom', 'flag']
for chunk in pd.read_csv(DATA_DIR / 'labevents.csv', usecols=lab_cols, chunksize=1_000_000):
    selected = chunk[chunk['itemid'].isin(lab_itemids)].copy()
    if len(selected):
        selected = selected.merge(d_labitems[['itemid', 'label']], on='itemid', how='left')
        selected = selected.merge(lab_catalog[['itemid', 'feature_name', 'type']], on='itemid', how='left')
        lab_chunks.append(selected)

In [127]:
lab_selected = pd.concat(lab_chunks, ignore_index=True)

lab_selected['charttime'] = pd.to_datetime(lab_selected['charttime'], errors='coerce')

lab_selected = lab_selected.merge(adm_pat_icu[['subject_id', 'hadm_id', 'stay_id', 'intime', 'outtime']], on=['subject_id', 'hadm_id'], how='inner')
lab_selected = lab_selected[(lab_selected['charttime'] >= lab_selected['intime']) &
                            (lab_selected['charttime'] <= lab_selected['outtime'])].copy()

lab_selected.to_csv(EXTRACT_DIR / 'lab_selected.csv', index=False)

print('lab_selected rows:', len(lab_selected))
print('lab_selected stays:', lab_selected['stay_id'].nunique())
print('lab_itemid counts to be extracted:', len(lab_catalog))
print('lab_itemid counts extracted:', lab_selected.itemid.nunique())
print('lab_feature counts:', lab_selected.feature_name.nunique())


lab_selected rows: 107288
lab_selected stays: 937
lab_itemid counts to be extracted: 41
lab_itemid counts extracted: 31
lab_feature counts: 28


## 4. EMAR 약물 추출


In [128]:
med_catalog = variable_catalog[variable_catalog['source_table'].eq('emar')].copy()
med_catalog.head()

,source_table,itemid,category,label,feature_name,unit_or_window,type,transform_note,binning
71,emar,NaN,NaN,*NF* Carbidopa-Levodopa (25-100),levodopa_related,NaN,binary,positive(1) if the observation window overla...,at least once
72,emar,NaN,NaN,Carbidopa-Levodopa (10-100),levodopa_related,NaN,binary,positive(1) if the observation window overla...,at least once
73,emar,NaN,NaN,Carbidopa-Levodopa (25-100),levodopa_related,NaN,binary,positive(1) if the observation window overla...,at least once
74,emar,NaN,NaN,Carbidopa-Levodopa (25-100) ODT,levodopa_related,NaN,binary,positive(1) if the observation window overla...,at least once
75,emar,NaN,NaN,Carbidopa-Levodopa (25-250),levodopa_related,NaN,binary,positive(1) if the observation window overla...,at least once


In [129]:
med_labels = med_catalog['label'].tolist()
med_catalog['medication_name'] = med_catalog['label'].astype('string').str.strip()

# stopped, not given 등 투약이 이루어지지 않은 경우들도 있으므로 투약한 경우에 해당하는 event_txt들만 추출
administration_event_txt = [
    'Administered', 'Delayed Administered', 'Administered in Other Location',
    'Administered Bolus from IV Drip', 'Partial Administered',
    'Started', 'Delayed Started', 'Started in Other Location', 'Restarted',
    'Applied', 'Applied in Other Location', 'Removed Existing / Applied New',
    'Removed Existing / Applied New in Other Location', 'Confirmed',
    'Confirmed in Other Location', 'Delayed Confirmed', 'Infusion Reconciliation',
    'Rate Change',
]

In [130]:
emar_cols = ['subject_id', 'hadm_id', 'emar_id', 'emar_seq', 'charttime', 'medication', 'event_txt']
emar = pd.read_csv(DATA_DIR / 'emar.csv', usecols=emar_cols)
emar['charttime'] = pd.to_datetime(emar['charttime'], errors='coerce')
emar['medication_name'] = emar['medication'].astype('string').str.strip()
emar = emar[emar['event_txt'].isin(administration_event_txt)].copy()

In [131]:
med_selected = emar.merge(med_catalog[['medication_name', 'feature_name']], on='medication_name', how='inner')
med_selected = med_selected.merge(adm_pat_icu[['subject_id', 'hadm_id', 'stay_id', 'intime', 'outtime']], on=['subject_id', 'hadm_id'], how='inner')
med_selected = med_selected[med_selected['charttime'].notna() & 
                            (med_selected['charttime'] >= med_selected['intime']) &
                            (med_selected['charttime'] <= med_selected['outtime'])].copy()

med_selected.to_csv(EXTRACT_DIR / 'medication_events.csv', index=False)

print('medication_events rows:', len(med_selected))
print('medication_events stays:', med_selected['stay_id'].nunique())
print('medication label counts to be extracted:', len(med_labels))
print('medication label counts extracted:', med_selected.medication_name.nunique())
print('medication feature counts:', med_selected.feature_name.nunique())

medication_events rows: 18205
medication_events stays: 675
medication label counts to be extracted: 137
medication label counts extracted: 77
medication feature counts: 11


## 5. PROCEDUREEVENTS 추출


In [132]:
procedure_catalog = variable_catalog[variable_catalog['source_table'].eq('procedureevents')].copy()
procedure_catalog['itemid'] = pd.to_numeric(procedure_catalog['itemid'], errors='coerce').astype('int')
procedure_catalog.head()

,source_table,itemid,category,label,feature_name,unit_or_window,type,transform_note,binning
69,procedureevents,225792,NaN,Invasive Ventilation,procedure_invasive_ventilation,NaN,binary,positive(1) if procedure interval (starttime-...,at least once
70,procedureevents,225794,NaN,Non-invasive Ventilation,procedure_noninvasive_ventilation,NaN,binary,positive(1) if procedure interval (starttime-...,at least once


In [133]:
procedure_itemids = procedure_catalog['itemid'].tolist()
procedure_labels  = procedure_catalog['label'].tolist()

# 큰 파일이므로 chunk 단위로 read
procedure_chunks = []
procedure_cols = ['subject_id', 'hadm_id', 'stay_id', 'starttime', 'endtime', 'itemid', 'value', 'valueuom', 'statusdescription']

for chunk in pd.read_csv(DATA_DIR / 'procedureevents.csv', usecols=procedure_cols, chunksize=500_000):
    selected = chunk[chunk['itemid'].isin(procedure_itemids)].copy()
    if len(selected):
        selected = selected.merge(d_items[['itemid', 'label', 'category']], on='itemid', how='left')
        selected = selected.merge(procedure_catalog[['itemid', 'feature_name', 'type']], on='itemid', how='left')
        procedure_chunks.append(selected)


In [134]:
procedure_selected = pd.concat(procedure_chunks, ignore_index=True)

procedure_selected['starttime'] = pd.to_datetime(procedure_selected['starttime'], errors='coerce')
procedure_selected['endtime']   = pd.to_datetime(procedure_selected['endtime'], errors='coerce')

procedure_selected = procedure_selected.merge(adm_pat_icu[['subject_id', 'hadm_id', 'stay_id', 'intime', 'outtime']], 
                                              on=['subject_id', 'hadm_id', 'stay_id'], how='inner')
procedure_selected = procedure_selected[(procedure_selected['starttime'] <= procedure_selected['outtime']) &
                                        (procedure_selected['endtime'] >= procedure_selected['intime'])].copy()

procedure_selected.to_csv(EXTRACT_DIR / 'procedure_selected.csv', index=False)

print('procedure_selected rows:', len(procedure_selected))
print('procedure_selected stays:', procedure_selected['stay_id'].nunique())
print('procedure_itemid counts to be extracted:', len(procedure_catalog))
print('procedure_itemid counts extracted:', procedure_selected.itemid.nunique())
print('procedure_feature counts:', procedure_selected.feature_name.nunique())

procedure_selected rows: 321
procedure_selected stays: 276
procedure_itemid counts to be extracted: 2
procedure_itemid counts extracted: 2
procedure_feature counts: 2


## 6. Long-format event table 저장

In [135]:
chart_long = chart_selected.copy()
chart_long['source_table'] = 'chartevents'
chart_long = chart_long[['source_table', 'subject_id', 'hadm_id', 'stay_id', 'charttime', 'itemid', 'label', 'feature_name', 'type', 'value', 'valuenum', 'valueuom']]

lab_long = lab_selected.copy()
lab_long['source_table'] = 'labevents'
lab_long['type'] = 'numeric'
lab_long = lab_long[['source_table', 'subject_id', 'hadm_id', 'stay_id', 'charttime', 'itemid', 'label', 'feature_name', 'type', 'value', 'valuenum', 'valueuom']]

med_long = med_selected.copy()
med_long['source_table'] = 'emar'
med_long['itemid'] = pd.NA
med_long['label'] = med_long['medication_name']
med_long['type'] = 'binary'
med_long['value'] = 1
med_long['valuenum'] = 1
med_long['valueuom'] = pd.NA
med_long = med_long[['source_table', 'subject_id', 'hadm_id', 'stay_id', 'charttime', 'itemid', 'label', 'feature_name', 'type', 'value', 'valuenum', 'valueuom']]

all_events_long = pd.concat([chart_long, lab_long, med_long], ignore_index=True)
all_events_long.to_csv(EXTRACT_DIR / 'all_events_long.csv', index=False)

print('all_events_long rows:', len(all_events_long))
print(all_events_long['source_table'].value_counts())


all_events_long rows: 805172
source_table
chartevents    679679
labevents      107288
emar            18205
Name: count, dtype: int64


In [136]:
all_events_long

,source_table,subject_id,hadm_id,stay_id,charttime,itemid,label,feature_name,type,value,valuenum,valueuom
0,chartevents,14979348,22895491.0,39414466,2139-08-30 23:50:00,220179,Non Invasive Blood Pressure systolic,nibp_sbp,numeric,170,170.0,mmHg
1,chartevents,14979348,22895491.0,39414466,2139-08-30 23:50:00,220180,Non Invasive Blood Pressure diastolic,nibp_dbp,numeric,80,80.0,mmHg
2,chartevents,14979348,22895491.0,39414466,2139-08-30 23:50:00,220181,Non Invasive Blood Pressure mean,nibp_mbp,numeric,102,102.0,mmHg
3,chartevents,14979348,22895491.0,39414466,2139-08-30 23:51:00,220045,Heart Rate,heart_rate,numeric,93,93.0,bpm
4,chartevents,14979348,22895491.0,39414466,2139-08-30 23:51:00,220210,Respiratory Rate,respiratory_rate,numeric,18,18.0,insp/min
...,...,...,...,...,...,...,...,...,...,...,...,...
805167,emar,14240547,25871919.0,30519369,2209-11-21 14:27:00,<NA>,Carbidopa-Levodopa (25-100),levodopa_related,binary,1,1.0,<NA>
805168,emar,14240547,25871919.0,30519369,2209-11-21 20:32:00,<NA>,Carbidopa-Levodopa (25-100),levodopa_related,binary,1,1.0,<NA>
805169,emar,14240547,25871919.0,30519369,2209-11-22 08:03:00,<NA>,Carbidopa-Levodopa (25-100),levodopa_related,binary,1,1.0,<NA>
805170,emar,14240547,25871919.0,30519369,2209-11-22 13:50:00,<NA>,Carbidopa-Levodopa (25-100),levodopa_related,binary,1,1.0,<NA>
